In [12]:
import json
import requests
from pathlib import Path

import duckdb
import polyline
from geopy.distance import geodesic

from to_gpx import to_gpx

In [13]:
DATA_BASE_PATH = Path("./data").resolve().absolute()
GRAPHHOPPER_BASE_URL = "http://localhost:8989"
GPS_ACCURACY = 50

In [14]:
gps_data_path = DATA_BASE_PATH / "gps_data.parquet"
ground_truth_path = DATA_BASE_PATH / "ground_truth_route.parquet"
newson_krumm_route_network_path = DATA_BASE_PATH / "road_network.parquet"

In [15]:
ground_truth_df = duckdb.query(
    f"""
    SELECT gt.edge_id, traversed, linestring 
    FROM '{ground_truth_path}' gt 
    JOIN '{newson_krumm_route_network_path}' nkr 
    ON gt.edge_id = nkr.edge_id
    WHERE gt.traversed = 1
    """
).to_df()

In [16]:
ground_truth_df["track_segs"] = ground_truth_df["linestring"].str.replace(
    r"^LINESTRING\(|\)$", "", regex=True
)
ground_truth_df["track_segs"] = (
    ground_truth_df["track_segs"].str.replace(",", ";").replace(r"\s+", " ", regex=True)
)
ground_truth_df["track_segs"] = ground_truth_df["track_segs"].str.split(";")
ground_truth_df["track_segs"] = ground_truth_df["track_segs"].apply(
    lambda x: [[float(coord) for coord in point.split()] for point in x]
)
ground_truth_df["track_segs"]

0      [[-122.349758148193, 47.6465114951134], [-122....
1      [[-122.34973937273, 47.6461198925972], [-122.3...
2      [[-122.356890141964, 47.6449799537659], [-122....
3      [[-122.356908917427, 47.6432499289513], [-122....
4      [[-122.356908917427, 47.6441296935081], [-122....
                             ...                        
396    [[-122.141988873482, 47.6346802711487], [-122....
397    [[-122.143048346043, 47.6342108845711], [-122....
398    [[-122.099888920784, 47.5678691267967], [-122....
399    [[-122.144249975681, 47.633650302887], [-122.1...
400    [[-122.149338126183, 47.6307910680771], [-122....
Name: track_segs, Length: 401, dtype: object

In [17]:
gps_df = duckdb.query(f"SELECT * FROM '{gps_data_path}'").to_df()
gps_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7531 entries, 0 to 7530
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   recorded_timestamp  7531 non-null   datetime64[us]
 1   lon                 7531 non-null   float64       
 2   lat                 7531 non-null   float64       
dtypes: datetime64[us](1), float64(2)
memory usage: 176.6 KB


In [18]:
gps_df.head()

,recorded_timestamp,lon,lat
0,2009-01-17 20:27:37,-122.107083,47.667483
1,2009-01-17 20:27:38,-122.107067,47.667500
2,2009-01-17 20:27:39,-122.107067,47.667500
3,2009-01-17 20:27:40,-122.107033,47.667517
4,2009-01-17 20:27:41,-122.106983,47.667533


In [19]:
points = gps_df[["lon", "lat", "recorded_timestamp"]].to_numpy()

gpx_path = to_gpx(points, DATA_BASE_PATH / "gps.gpx")

In [20]:
def request_map_matching(point_gpx_path: Path, output_path: Path) -> str:
    url = f"{GRAPHHOPPER_BASE_URL}/match?profile=car&gps_accuracy={GPS_ACCURACY}&type=json&locale=pt_BR"
    headers = {
        "Content-Type": "application/gpx+xml",
    }

    with open(point_gpx_path, "rb") as f:
        body = f.read()

    req = requests.post(
        url,
        headers=headers,
        data=body,
    )

    req.raise_for_status()

    with open(output_path, "wb") as f:
        f.write(req.content)

    print(f"Map matching result saved to {output_path}")

    return json.loads(req.content)["paths"][0]["points"]

In [21]:
graphhopper_result = request_map_matching(
    point_gpx_path=gpx_path, output_path=DATA_BASE_PATH / "map_matched.json"
)
graphhopper_result

Map matching result saved to /home/jose_edsouza/Documentos/Faculdade/TCC/repo/dataset/newson-krumm/data/map_matched.json


'w`}aHd`hhVC?Ii@?_HAaH?yGAYSo@Ie@cAlBa@z@_@|@]|@[`Ac@xAYz@Qd@SX[V[RmCpAcAj@_@ZU\\Qd@Kf@Ad@@p@Hr@H^Td@RRTR`Aj@pFzC\\TPNt@~@h@`Ad@dA^fA\\zA`@vBVpBLrBHjD@tBJdBKl]AbOBbW@fBFfATxDT|B\\`Cr@~Cr@lC|@dClAnCt@tAjAdBr@v@tAnAtB|AzAv@tNlGrD|BhJhEfDxAxAx@`ElB|EhCVFl@cJVwEL}AV}E@w@CyAAk@i@oIk@iLw@sNGi@Oc@uCiHyBkFkCiEoBsFu@{BKSfByBxAyAzEqFzAaB~BuBdAs@z@c@l@YnA_@p@QvIaAl@Kt@UGg@AIBUJWf@_AZq@Ro@BQVeBpCcSn@kETgAd@{ATk@l@gAp@}@z@w@bAk@d@OhAWjNaBXId@]LQbBuC`@y@nIgOtCoFd@gApBmFf@kAvA}DtCgH~F{LVk@nCkFh@_AhA{At@}@fKaLvAgBbDeD^g@PMtAuArAgAlAu@fAk@xAm@hA]jAWn@G`BKh@?hADlAJjAXtAb@v@\\p@`@|D~CnAr@hH|Cn@^dCzAbCnBdBjBlClD~AnC`BhDz@|BvCtJb@nAj@lAb@x@p@x@zInKr@dAPZp@zAlIhUrA`DrAxCn@nArCdFn@~@r@~@zAxAdH~El@^^Vh@`@pAfAr@t@zFhHt@j@XLVHZHZBb@?x@MlNsDh@Qh[iIVEXEf@?pRnB~@Nj@Nt@ZlFfCt@Nx@D|m@cDrCMP?j@FdAVjZlIXN|A`@`ANnFf@TAXAj@OPMVSTWJO\\q@Lc@Fa@NoALu@\\eATg@l@kAx@cAlCuCrCuCnGwGhCoC^g@Zs@To@RaATkB`AkHd@wCLq@Ni@x@iCt@}BTe@Ra@TYj@k@`@W\\OnDmAiA^hA_@TCbAYJ?NFHFF?F?DCDEDKLIZMp@_@NCN@BDHFN?FERU\\MpAQvD]tD[RIh@NdAAn@DHDFHBJ@PCf@y

In [22]:
graphhopper_result

'w`}aHd`hhVC?Ii@?_HAaH?yGAYSo@Ie@cAlBa@z@_@|@]|@[`Ac@xAYz@Qd@SX[V[RmCpAcAj@_@ZU\\Qd@Kf@Ad@@p@Hr@H^Td@RRTR`Aj@pFzC\\TPNt@~@h@`Ad@dA^fA\\zA`@vBVpBLrBHjD@tBJdBKl]AbOBbW@fBFfATxDT|B\\`Cr@~Cr@lC|@dClAnCt@tAjAdBr@v@tAnAtB|AzAv@tNlGrD|BhJhEfDxAxAx@`ElB|EhCVFl@cJVwEL}AV}E@w@CyAAk@i@oIk@iLw@sNGi@Oc@uCiHyBkFkCiEoBsFu@{BKSfByBxAyAzEqFzAaB~BuBdAs@z@c@l@YnA_@p@QvIaAl@Kt@UGg@AIBUJWf@_AZq@Ro@BQVeBpCcSn@kETgAd@{ATk@l@gAp@}@z@w@bAk@d@OhAWjNaBXId@]LQbBuC`@y@nIgOtCoFd@gApBmFf@kAvA}DtCgH~F{LVk@nCkFh@_AhA{At@}@fKaLvAgBbDeD^g@PMtAuArAgAlAu@fAk@xAm@hA]jAWn@G`BKh@?hADlAJjAXtAb@v@\\p@`@|D~CnAr@hH|Cn@^dCzAbCnBdBjBlClD~AnC`BhDz@|BvCtJb@nAj@lAb@x@p@x@zInKr@dAPZp@zAlIhUrA`DrAxCn@nArCdFn@~@r@~@zAxAdH~El@^^Vh@`@pAfAr@t@zFhHt@j@XLVHZHZBb@?x@MlNsDh@Qh[iIVEXEf@?pRnB~@Nj@Nt@ZlFfCt@Nx@D|m@cDrCMP?j@FdAVjZlIXN|A`@`ANnFf@TAXAj@OPMVSTWJO\\q@Lc@Fa@NoALu@\\eATg@l@kAx@cAlCuCrCuCnGwGhCoC^g@Zs@To@RaATkB`AkHd@wCLq@Ni@x@iCt@}BTe@Ra@TYj@k@`@W\\OnDmAiA^hA_@TCbAYJ?NFHFF?F?DCDEDKLIZMp@_@NCN@BDHFN?FERU\\MpAQvD]tD[RIh@NdAAn@DHDFHBJ@PCf@y

In [23]:
grapphopper_trajectory = polyline.decode(graphhopper_result)

grapphopper_trajectory

[(47.66748, -122.10707),
 (47.6675, -122.10707),
 (47.66755, -122.10686),
 (47.66755, -122.10542),
 (47.66756, -122.10397),
 (47.66756, -122.10256),
 (47.66757, -122.10243),
 (47.66767, -122.10219),
 (47.66772, -122.102),
 (47.66806, -122.10255),
 (47.66823, -122.10285),
 (47.66839, -122.10316),
 (47.66854, -122.10347),
 (47.66868, -122.1038),
 (47.66886, -122.10425),
 (47.66899, -122.10455),
 (47.66908, -122.10474),
 (47.66918, -122.10487),
 (47.66932, -122.10499),
 (47.66946, -122.10509),
 (47.67017, -122.1055),
 (47.67051, -122.10572),
 (47.67067, -122.10586),
 (47.67078, -122.10601),
 (47.67087, -122.1062),
 (47.67093, -122.1064),
 (47.67094, -122.10659),
 (47.67093, -122.10684),
 (47.67088, -122.1071),
 (47.67083, -122.10726),
 (47.67072, -122.10745),
 (47.67062, -122.10755),
 (47.67051, -122.10765),
 (47.67018, -122.10787),
 (47.66897, -122.10865),
 (47.66882, -122.10876),
 (47.66873, -122.10884),
 (47.66846, -122.10916),
 (47.66825, -122.10949),
 (47.66806, -122.10984),
 (47.667

In [24]:
real_trajectory = (
    ground_truth_df["track_segs"].explode().apply(lambda x: (x[1], x[0])).tolist()
)
real_trajectory

[(47.6465114951134, -122.349758148193),
 (47.6469111442566, -122.349768877029),
 (47.6461198925972, -122.34973937273),
 (47.6465114951134, -122.349758148193),
 (47.6449799537659, -122.356890141964),
 (47.6458194851875, -122.356879413128),
 (47.6432499289513, -122.356908917427),
 (47.6441296935081, -122.356908917427),
 (47.6441296935081, -122.356908917427),
 (47.6449799537659, -122.356890141964),
 (47.6414394378662, -122.356919646263),
 (47.6432499289513, -122.356908917427),
 (47.639591395855, -122.356919646263),
 (47.6414394378662, -122.356919646263),
 (47.6383709907532, -122.356930375099),
 (47.639591395855, -122.356919646263),
 (47.6371800899506, -122.356938421726),
 (47.6383709907532, -122.356930375099),
 (47.63671875, -122.356935739517),
 (47.6371800899506, -122.356938421726),
 (47.6492205262184, -122.349768877029),
 (47.6495799422264, -122.349790334702),
 (47.6495799422264, -122.349790334702),
 (47.6501592993736, -122.349819839001),
 (47.6501592993736, -122.349819839001),
 (47.650

In [25]:
len(real_trajectory), len(grapphopper_trajectory)

(1401, 1214)

In [26]:
def calculate_distance(trajectory: list[tuple[float, float]]) -> float:
    return sum(
        geodesic(trajectory[i], trajectory[i + 1]).meters
        for i in range(len(trajectory) - 1)
    )

In [27]:
real_length = calculate_distance(real_trajectory)
print(f"Comprimento da trajetória real: {real_length:.2f} metros")

Comprimento da trajetória real: 372237.90 metros


In [28]:
graphhopper_length = calculate_distance(grapphopper_trajectory)

print(f"Comprimento da trajetória do GraphHopper: {graphhopper_length:.2f} metros")

Comprimento da trajetória do GraphHopper: 84110.58 metros


In [29]:
graphhopper_length / real_length

0.22595920348332568